# Compliance and PII Scanning

This notebook demonstrates Zipminator's compliance tooling: PII detection,
GDPR-aligned anonymization, and audit trail generation.

These features help organizations meet requirements under GDPR, HIPAA, and CCPA.

In [1]:
# PII Scanning on sample data
import pandas as pd
from zipminator.scanner import QuantumReadinessScanner

scanner = QuantumReadinessScanner(sensitivity="high")

# Sample dataset with mixed PII
patient_data = pd.DataFrame({
    "patient_name": ["Alice Johnson", "Bob Smith", "Carol Williams"],
    "email": ["alice@hospital.org", "bob.smith@gmail.com", "carol@clinic.no"],
    "ssn": ["123-45-6789", "987-65-4321", "456-78-9012"],
    "diagnosis": ["Hypertension", "Type 2 Diabetes", "Asthma"],
    "phone": ["555-123-4567", "555-987-6543", "555-246-8135"],
    "dob": ["1990-03-15", "1972-08-22", "1998-11-04"],
})

print("Input Data:")
print(patient_data.to_string(index=False))
print()

# Run PII scan
findings = scanner.scan(patient_data)

print(f"PII Scan Results: {len(findings)} findings")
print("=" * 60)
for f in findings:
    print(f"  Column: {f.field:<15} Type: {f.pii_type:<10} Confidence: {f.confidence:.0%}")

# Generate report
report = scanner.report(findings)
print("\n" + report)

TypeError: QuantumReadinessScanner.__init__() got an unexpected keyword argument 'sensitivity'

In [ ]:
# GDPR-aligned anonymization
from zipminator.anonymizer import AdvancedAnonymizer

anonymizer = AdvancedAnonymizer(subscription_tier="enterprise")

# Art. 89 research exemption: use differential privacy (L8)
research_data = anonymizer.anonymize(patient_data, level=8, epsilon=0.5)

print("GDPR Art. 89 - Research Exemption (L8 Differential Privacy, epsilon=0.5):")
print(research_data.to_string(index=False))
print()

# HIPAA Safe Harbor: use data suppression (L6) to remove identifiers
safe_harbor = anonymizer.anonymize(patient_data, level=6)

print("HIPAA Safe Harbor (L6 Data Suppression):")
print(safe_harbor.to_string(index=False))
print()

# Art. 17 Right to Erasure preparation: identify what to delete
pii_columns = anonymizer.detect_pii_columns(patient_data)
print(f"Columns flagged for potential erasure (Art. 17): {pii_columns}")

In [ ]:
# Audit trail generation
import json
from datetime import datetime, timezone
from hashlib import sha3_256

# Simulate audit trail entries for the operations above
audit_entries = [
    {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "operation": "pii_scan",
        "input_hash": sha3_256(patient_data.to_csv().encode()).hexdigest()[:16],
        "findings_count": len(findings),
        "sensitivity": "high",
        "user": "analyst@corp.com",
    },
    {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "operation": "anonymize",
        "level": 8,
        "input_hash": sha3_256(patient_data.to_csv().encode()).hexdigest()[:16],
        "output_hash": sha3_256(research_data.to_csv().encode()).hexdigest()[:16],
        "parameters": {"epsilon": 0.5},
        "user": "analyst@corp.com",
    },
    {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "operation": "anonymize",
        "level": 6,
        "input_hash": sha3_256(patient_data.to_csv().encode()).hexdigest()[:16],
        "output_hash": sha3_256(safe_harbor.to_csv().encode()).hexdigest()[:16],
        "parameters": {},
        "user": "analyst@corp.com",
    },
]

print("Audit Trail")
print("=" * 70)
for entry in audit_entries:
    print(json.dumps(entry, indent=2))
    print()